# Midpoint MF-GP Comparison

Compare the baseline MF-GP run against LF augmentation with synthetic midpoint theta values.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_mfgp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_mfgp"))
if str(REPO_ROOT / "theta_augmentations" / "midpoint") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "theta_augmentations" / "midpoint"))

from build_midpoint_augmented_lf import default_paths
from mfgp_clean_pipeline import load_runtime_config, run_mfgp_transform_suite

PATHS = default_paths()
ARTIFACT_DIR = PATHS.artifact_dir
BASELINE_CONFIG = ARTIFACT_DIR / "settings_baseline.yaml"
MIDPOINT_CONFIG = ARTIFACT_DIR / "settings_midpoint.yaml"
BASELINE_CNP_CSV = PATHS.training_cnp_csv
MIDPOINT_CNP_CSV = ARTIFACT_DIR / "cnp_augmented_midpoint_training.csv"
VALIDATION_CNP_CSV = PATHS.validation_cnp_csv
ITERATION = 0
GRID_POINTS = 120
PREDICT_CHUNK_SIZE = 20000
RANDOM_STATE = 42
TARGET_TRANSFORMS = ["linear", "log_hf", "log_lf", "log_both"]


## Inputs

In [ ]:
artifact_table = pd.DataFrame({
    "experiment": ["baseline", "midpoint"],
    "lf_mode": ["original only", "original + midpoint theta LF"],
    "config_path": [str(BASELINE_CONFIG), str(MIDPOINT_CONFIG)],
    "training_cnp_csv": [str(BASELINE_CNP_CSV), str(MIDPOINT_CNP_CSV)],
})
display(artifact_table)

runtime = load_runtime_config(BASELINE_CONFIG)
display(pd.DataFrame({
    "field": ["version", "theta_headers", "theta_min", "theta_max", "out_dir_cnp", "out_dir_mfgp"],
    "value": [runtime.version, ", ".join(runtime.theta_headers), runtime.theta_min, runtime.theta_max, str(runtime.out_dir_cnp), str(runtime.out_dir_mfgp)],
}))


## Run MF-GP

In [ ]:
EXPERIMENTS = {
    "baseline": (BASELINE_CONFIG, BASELINE_CNP_CSV),
    "midpoint": (MIDPOINT_CONFIG, MIDPOINT_CNP_CSV),
}

all_mfgp_results = {}
for experiment_name, (config_path, cnp_csv) in EXPERIMENTS.items():
    all_mfgp_results[experiment_name] = run_mfgp_transform_suite(
        config_path=config_path,
        cnp_csv=cnp_csv,
        validation_csv=VALIDATION_CNP_CSV,
        transforms=TARGET_TRANSFORMS,
        iteration=ITERATION,
        grid_points_per_axis=GRID_POINTS,
        random_state=RANDOM_STATE,
        predict_chunk_size=PREDICT_CHUNK_SIZE,
        verbose=True,
    )

EXPERIMENT_TITLES = {
    "linear": "Normal MF-GP",
    "log_hf": "Log HF: emulate log10(y_raw)",
    "log_lf": "Log LF: use log10(y_cnp)",
    "log_both": "Log HF + Log LF: use log10(y_raw) and log10(y_cnp)",
}

PLOT_SELECTIONS = {
    "linear": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_hf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_lf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_both": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
}

def _display_path_artifact(title: str, artifact_path: str) -> None:
    print(title)
    print(f"Open manually: {artifact_path}")

def _display_image_artifact(title: str, artifact_path: str) -> None:
    print(title)
    display(Image(filename=str(artifact_path)))

def display_selected_mfgp_plots(experiment_name: str, mode: str) -> None:
    result = all_mfgp_results[experiment_name][mode]
    print(f"{experiment_name}: {EXPERIMENT_TITLES[mode]}\n")
    for title, attr_name in PLOT_SELECTIONS[mode]:
        artifact_path = getattr(result, attr_name, None)
        if not artifact_path:
            continue
        if str(artifact_path).lower().endswith(".html"):
            _display_path_artifact(title, str(artifact_path))
        else:
            _display_image_artifact(title, str(artifact_path))
        print()


## Summary

In [ ]:
summary_rows = []
artifact_rows = []
preview_frames = []
for experiment_name, per_mode in all_mfgp_results.items():
    for mode, result in per_mode.items():
        metrics = json.loads(Path(result.metrics_json).read_text())
        summary_rows.append({
            "experiment": experiment_name,
            "mode": mode,
            "rmse_hf": metrics.get("rmse_hf"),
            "mae_hf": metrics.get("mae_hf"),
            "r2_hf": metrics.get("r2_hf"),
            "n_lf": metrics.get("n_lf"),
            "n_hf": metrics.get("n_hf"),
            "rho": metrics.get("rho"),
            "target_transform": metrics.get("target_transform"),
            "target_transform_eps": metrics.get("target_transform_eps"),
        })
        for label, attr_name in PLOT_SELECTIONS[mode]:
            artifact_path = getattr(result, attr_name, None)
            if artifact_path:
                artifact_rows.append({
                    "experiment": experiment_name,
                    "mode": mode,
                    "artifact": label,
                    "path": str(artifact_path),
                })
        for attr_name in ["cnp_csv", "model_json", "metrics_json", "prediction_csv", "grid_csv"]:
            artifact_rows.append({
                "experiment": experiment_name,
                "mode": mode,
                "artifact": attr_name,
                "path": str(getattr(result, attr_name)),
            })
        preview = pd.read_csv(result.prediction_csv).head()
        preview.insert(0, "mode", mode)
        preview.insert(0, "experiment", experiment_name)
        preview_frames.append(preview)

display(pd.DataFrame(summary_rows))
display(pd.DataFrame(artifact_rows))
for preview in preview_frames:
    experiment_name = preview.iloc[0]["experiment"]
    mode = preview.iloc[0]["mode"]
    print(f"=== {experiment_name} | {mode} prediction CSV preview ===")
    display(preview)


## Baseline

### linear

In [ ]:
display_selected_mfgp_plots("baseline", "linear")


### log_hf

In [ ]:
display_selected_mfgp_plots("baseline", "log_hf")


### log_lf

In [ ]:
display_selected_mfgp_plots("baseline", "log_lf")


### log_both

In [ ]:
display_selected_mfgp_plots("baseline", "log_both")


## Midpoint

### linear

In [ ]:
display_selected_mfgp_plots("midpoint", "linear")


### log_hf

In [ ]:
display_selected_mfgp_plots("midpoint", "log_hf")


### log_lf

In [ ]:
display_selected_mfgp_plots("midpoint", "log_lf")


### log_both

In [ ]:
display_selected_mfgp_plots("midpoint", "log_both")


In [ ]:
theta_group_dirs = []
for experiment_name, per_mode in all_mfgp_results.items():
    for mode, result in per_mode.items():
        theta_group_dirs.append({
            "experiment": experiment_name,
            "mode": mode,
            "theta_group_plot_dir": str(result.theta_group_plot_dir) if result.theta_group_plot_dir else None,
        })
display(pd.DataFrame(theta_group_dirs))
